In [ ]:
%run /Workspace/consolidated_pipeline/1_setup/utilities
print(bronze_schema, silver_schema, gold_schema)

In [ ]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'customers', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

base_path = f's3:/sports-consolidated-dp/{data_source}.csv'

In [ ]:
df = (
    spark.read.format('csv')
    .option('header', True)
    .option('inferSchema', True)
    .load(base_path)
    .withColumn('read_timestamp', F.current_timestamp())
    .select('*', '_meta.file_name', '_meta.file_size')
)

df.printSchema()

In [ ]:
root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)

In [1]:
display(df.limit(10))

customer_id,customer_name,city,read_timestamp,file_name,file_size
,FitFuel Market,New Delhi,2025-11-23T11:52:51.119Z,customers.csv,1404
789301,Athlete's Choice Store,Bengaluru,2025-11-23T11:52:51.119Z,customers.csv,1404
789303,Athlete's Choice Store,New Delhi,2025-11-23T11:52:51.119Z,customers.csv,1404
789101,Endurance Foods,Bengalore,2025-11-23T11:52:51.119Z,customers.csv,1404
789102,Endurance Foods,Hyderabad,2025-11-23T11:52:51.119Z,customers.csv,1404
789103,Endurance Foods,New Delhi,2025-11-23T11:52:51.119Z,customers.csv,1404
789121,HydroBoost Nutrition,Hyderabad,2025-11-23T11:52:51.119Z,customers.csv,1404
789122,HydroBoost Nutrition,New Delhi,2025-11-23T11:52:51.119Z,customers.csv,1404


In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [ ]:
dbutils.widgets.text('data_source', 'customers' 'Data Source')
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source)
df_bronze = spark.sql(f'SELECT * FROM {catalog}.{bronze_schema}.{data_source};')

# View duplicates
df_duplicates = df_bronze.groupBy('customer_id').count().filter(F.col('count') > 1)
display(df_duplicates)

print('Rows before duplicates dropped: ' df_bronze.count()) 
df_silver = df_bronze.dropDuplicates(['customer_id'])
print('Rows after duplicates dropped: ', df_silver.count())

# Rows before duplicates dropped:  39
# Rows after duplicates dropped:  35

In [ ]:
# Trim spaces in customer_name

dispaly(
    df_bronze.filter(F.col('customer_name') != F.trim(F.col('customer_name')))
)

### Rows where `customer_name` has extra spaces

The first column in the paste had no header. Those values are customer ids.

| customer_id | customer_name | city | read_timestamp | file_name | file_size |
| --- | --- | --- | --- | --- | --- |
| 789121 | HydroBoost Nutrition | Hyderabad | 2025-11-23T11:53:03.643Z | customers.csv | 1404 |
| 789401 | SprintX nutrition | Bengaluru | 2025-11-23T11:53:03.643Z | customers.csv | 1404 |
| 789420 | ZenAthlete foods | null | 2025-11-23T11:53:03.643Z | customers.csv | 1404 |
| 789421 | ZenAthlete Foods | Hyderbad | 2025-11-23T11:53:03.643Z | customers.csv | 1404 |
| 789521 | PrimeFuel Nutrition | null | 2025-11-23T11:53:03.643Z | customers.csv | 1404 |
| 789702 | StaminaX Store | Hyderabad | 2025-11-23T11:53:03.643Z | customers.csv | 1404 |


In [ ]:
df_silver.select('city).distinct().show()

city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': ;Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',
    
    'NewDheli': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}

allowed = ['Bengaluru', 'Hyderabad', 'New Delhi']

df_silver = (
    df_silver
    .replace(city_mapping, subset=['city'])
    .withColumn(
        'city',
        F.when(F.col('city').isNull(), None)
         .when(F.col('city').isin(allowed), F.col('city'))
         .otherwise(None)
    )
)

In [ ]:
# Title case fix

df_silver.select('customer_name').distinct.show()

df_silver = df_silver.withColumn(
    'customer_name',
    F.when(F.col('customer_name').isNull(), None)
    .otherwise(F.initcap('customer_name'))
)

# Sanity check
df_silver.select('customer_name').distinct().show()

In [ ]:
# Handling missing cities

df_silver.filter(F.col('city').isNull()).show(truncate=False)

null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(F.col('customer_name').isin(null_customer_names)).show(truncate=False)

In [ ]:
# Business confirmation Note: City corrections confirmed by business team
customer_city_fix = {
    # Sprintx Nutition
    789403: 'New Delhi',

    # Zenathlete Foods
    789420: 'Bengaluru',

    # Primefuel Nutrition
    789521: 'Hyderabad',

    # Recovery Lane
    789603: 'Hyderabad'
}

df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ['customer_id', 'fixed_city']-=
)

display(df_fix)

In [ ]:
df_silver = (
    df_silver
    .join(df_fix, 'customer_id', 'left')
    .withColumn(
        'city',
        F.coalesce('city', 'fixed_city') # Replace null with fixed city
    )
    .drop('fixed_city')
)

In [ ]:
# Convert customer_id to string
df_silver = df_silver.withColumn('customer_id', F.col('customer_id').cast('string'))
print(df_silver.printSchema())

In [ ]:
df_silver = (
    df_silver
    .withColumn(
        'customer',
        F.concat_ws('-', 'customer_name', F.coalesce(F.col('city'), F.lit('unknown')))

        # Static attribute aligned with parent data model
        .withColumn('market', F.lit('India'))
        .withColumn('platform', F.lit('Sports Bar'))
        .withColumn('channel', F.lit('Acquisition'))
    )
)

In [ ]:
df_silver.write\
    .format('delta') \
    .option('delta.enableChangeDataFeed', 'true') \
    .option('mergeSchema', 'true') \
    .mode('overwrite') \
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

In [ ]:
df_silver = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.{data_source};')

df_gold = df_silver.select('customer_id', 'customer_name', 'city', 'customer', 'market', 'platform', 'channel')

In [ ]:
df_gold.write\
    .format('delta') \
    .options('delta.enableChangeDataFeed', 'true') \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{gold_schema}.db_dim_{data_source}')

In [ ]:
# ---> MERGIN DATA SOURCE WITH PARENT <---
# ---> UPSERT APPROACH <---
delta_table = DeltaTable.forName(spark, 'fmcg.gold.dim_customers')
df_child_customers = spark.table('fmcg.gold.sb_dim_customers').select(
    'customer',
    'market',
    'platform',
    'channel'
)

delta_table.alias('target').merge(
    source=df_child_customers.alias('source'),
    condition='target.customer_code = source.customer_code'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()